# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/phinapenteriani/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Helper used throughout this notebook to report TFLite file sizes.
def file_size_kb(path):
    """Return file size in kilobytes (1024 bytes)."""
    return os.path.getsize(path) / 1024.0

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-20 20:13:38.493462: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.

X = df[feature_names].values
y = df["Class"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique classes:", np.unique(y))

X shape: (178, 13)
y shape: (178,)
Unique classes: [0 1 2]


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("y_train:", y_train.shape, " y_test:", y_test.shape)

X_train: (124, 13)  X_test: (54, 13)
y_train: (124,)  y_test: (54,)


In [8]:
# Step 3: Use StandardScaler to normalize the features

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled  = scaler.transform(X_test).astype(np.float32)

print("X_train_scaled mean per feature (should be ~0):", np.round(X_train_scaled.mean(axis=0), 3))
print("X_train_scaled std  per feature (should be ~1):", np.round(X_train_scaled.std(axis=0), 3))

X_train_scaled mean per feature (should be ~0): [ 0.  0.  0. -0. -0.  0.  0. -0.  0.  0.  0. -0.  0.]
X_train_scaled std  per feature (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [9]:
# Step 4: One-hot encode y_train and y_test.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat  = to_categorical(y_test,  num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape :", y_test_cat.shape)
print("Example one-hot row:", y_train_cat[0])

y_train_cat shape: (124, 3)
y_test_cat shape : (54, 3)
Example one-hot row: [1. 0. 0.]


In [10]:
# Step 5: Define the baseline DNN.

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax'),
])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
# Step 6: Compile and train for 20 epochs.

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=2,
)

Epoch 1/20
13/13 - 1s - loss: 1.0622 - accuracy: 0.4646 - val_loss: 0.9247 - val_accuracy: 0.6000 - 1s/epoch - 81ms/step
Epoch 2/20
13/13 - 0s - loss: 0.8118 - accuracy: 0.7879 - val_loss: 0.7161 - val_accuracy: 0.8400 - 56ms/epoch - 4ms/step
Epoch 3/20
13/13 - 0s - loss: 0.6173 - accuracy: 0.9293 - val_loss: 0.5535 - val_accuracy: 0.8800 - 50ms/epoch - 4ms/step
Epoch 4/20
13/13 - 0s - loss: 0.4529 - accuracy: 0.9596 - val_loss: 0.4230 - val_accuracy: 0.8800 - 48ms/epoch - 4ms/step
Epoch 5/20
13/13 - 0s - loss: 0.3253 - accuracy: 0.9697 - val_loss: 0.3270 - val_accuracy: 0.9200 - 49ms/epoch - 4ms/step
Epoch 6/20
13/13 - 0s - loss: 0.2355 - accuracy: 0.9697 - val_loss: 0.2606 - val_accuracy: 0.9200 - 50ms/epoch - 4ms/step
Epoch 7/20
13/13 - 0s - loss: 0.1767 - accuracy: 0.9798 - val_loss: 0.2130 - val_accuracy: 0.9200 - 60ms/epoch - 5ms/step
Epoch 8/20
13/13 - 0s - loss: 0.1362 - accuracy: 0.9899 - val_loss: 0.1828 - val_accuracy: 0.9200 - 53ms/epoch - 4ms/step
Epoch 9/20
13/13 - 0s - l

In [12]:
# Step 7: Evaluate the trained baseline.

train_loss, train_acc = model.evaluate(X_train_scaled, y_train_cat, verbose=0)
test_loss,  test_acc  = model.evaluate(X_test_scaled,  y_test_cat,  verbose=0)
print(f"Baseline training accuracy: {train_acc:.4f}")
print(f"Baseline test     accuracy: {test_acc:.4f}")

y_pred_base = np.argmax(model.predict(X_test_scaled, verbose=0), axis=1)
print("\nClassification report (baseline):")
print(classification_report(y_test, y_pred_base, digits=4))
print("Confusion matrix (baseline):")
print(confusion_matrix(y_test, y_pred_base))

Baseline training accuracy: 0.9839
Baseline test     accuracy: 0.9815

Classification report (baseline):
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        18
           1     1.0000    0.9524    0.9756        21
           2     0.9375    1.0000    0.9677        15

    accuracy                         0.9815        54
   macro avg     0.9792    0.9841    0.9811        54
weighted avg     0.9826    0.9815    0.9816        54

Confusion matrix (baseline):
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


In [13]:
# Step 8: Convert to TFLite (float32) and report size.

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_base = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_base)

print(f"Baseline (float32) TFLite size: {file_size_kb('model_base.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/p8/hgvq1dqj20ncmtw6s7ys78f80000gn/T/tmp0tprwwkq/assets


INFO:tensorflow:Assets written to: /var/folders/p8/hgvq1dqj20ncmtw6s7ys78f80000gn/T/tmp0tprwwkq/assets


Baseline (float32) TFLite size: 14.07 KB


2026-05-20 20:13:45.475270: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 20:13:45.475297: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 20:13:45.475790: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/p8/hgvq1dqj20ncmtw6s7ys78f80000gn/T/tmp0tprwwkq
2026-05-20 20:13:45.477326: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 20:13:45.477354: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/p8/hgvq1dqj20ncmtw6s7ys78f80000gn/T/tmp0tprwwkq
2026-05-20 20:13:45.482244: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-20 20:13:45.484897: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 20:13:45.561488: I tensorflow/cc/saved_model/loader.

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size."""

    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type  = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert and save.
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    y_pred = []
    for i in range(len(X_test)):
        x = X_test[i:i + 1].astype(np.float32)

        if input_details["dtype"] == np.int8:
            in_scale, in_zp = input_details["quantization"]
            x = np.round(x / in_scale + in_zp).astype(np.int8)

        interpreter.set_tensor(input_details["index"], x)
        interpreter.invoke()
        out = interpreter.get_tensor(output_details["index"])

        if output_details["dtype"] == np.int8:
            out_scale, out_zp = output_details["quantization"]
            out = (out.astype(np.float32) - out_zp) * out_scale

        y_pred.append(int(np.argmax(out)))

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")
    print(f"{quant_type.upper()} accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"\nClassification report ({quant_type}):")
    print(classification_report(y_true, y_pred, digits=4))
    print(f"Confusion matrix ({quant_type}):")
    print(confusion_matrix(y_true, y_pred))
    return y_pred

In [ ]:
# Step 5: Generate and evaluate the three quantized TFLite models.

_ = quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')
_ = quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8',    'model_int8.tflite')
_ = quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')

print("\n--- Part (b) Size Summary ---")
print(f"Baseline (float32)  : {file_size_kb('model_base.tflite'):.2f} KB")
print(f"Dynamic range       : {file_size_kb('model_dynamic.tflite'):.2f} KB")
print(f"Full integer (int8) : {file_size_kb('model_int8.tflite'):.2f} KB")
print(f"Float16             : {file_size_kb('model_float16.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/p8/hgvq1dqj20ncmtw6s7ys78f80000gn/T/tmpnywxug9c/assets


INFO:tensorflow:Assets written to: /var/folders/p8/hgvq1dqj20ncmtw6s7ys78f80000gn/T/tmpnywxug9c/assets


## Problem 1 - Part (c)

### Pruning

In [ ]:
# Step 1: Define the pruning schedule.

batch_size   = 8
prune_epochs = 10
num_train_samples = X_train_scaled.shape[0]
end_step = int(np.ceil(num_train_samples / batch_size) * prune_epochs)
print("end_step:", end_step)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.50,
    final_sparsity=0.70,
    begin_step=0,
    end_step=end_step,
)

In [ ]:
# Step 2: Build the pruned model (wrap each Dense with prune_low_magnitude).

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(num_features,)),
        pruning_schedule=pruning_schedule,
    ),
    prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule,
    ),
    prune_low_magnitude(
        Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule,
    ),
])
pruned_model.summary()

In [ ]:
# Step 3: Compile and train the pruned model.

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

pruned_history = pruned_model.fit(
    X_train_scaled, y_train_cat,
    epochs=prune_epochs,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=2,
)

In [ ]:
# Step 4: Strip pruning wrappers and convert to TFLite.

stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
# Default optimizations help sparse weights compress better on disk.
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_pruned = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned)

print(f"Pruned TFLite size: {file_size_kb('model_pruned.tflite'):.2f} KB")

In [ ]:
# Step 5: Evaluate the stripped pruned model.

y_pred_pruned = np.argmax(stripped_model.predict(X_test_scaled, verbose=0), axis=1)
print(f"Pruned model accuracy: {accuracy_score(y_test, y_pred_pruned):.4f}")
print("\nClassification report (pruned):")
print(classification_report(y_test, y_pred_pruned, digits=4))
print("Confusion matrix (pruned):")
print(confusion_matrix(y_test, y_pred_pruned))

## Problem 1 - Part (d)

### Knowledge Distillation

In [ ]:
# Step 1: Define the smaller student model.

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax'),
])
student_model.summary()

In [ ]:
# Step 2: Teacher soft labels.

teacher_preds_soft = model.predict(X_train_scaled, verbose=0)
print("teacher_preds_soft shape:", teacher_preds_soft.shape)
print("Example soft label:", np.round(teacher_preds_soft[0], 4))

In [ ]:
# Step 3: Combined labels + distillation loss.

# (a) Concatenate hard and soft labels along axis=1.
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1).astype(np.float32)
print("Combined label shape:", y_train_combined.shape)  # (n, 6) for 3-class problem

# (b) Custom distillation loss with weight alpha = 0.5.
alpha = 0.5

def distillation_loss(y_true_combined, y_pred):
    # Split combined labels into hard and soft components.
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    cce = tf.keras.losses.categorical_crossentropy
    loss_hard = cce(y_true_hard, y_pred)
    loss_soft = cce(y_true_soft, y_pred)

    return alpha * loss_hard + (1.0 - alpha) * loss_soft

In [ ]:
# Step 4: Compile and train the student using distillation loss.

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy'],
)

student_history = student_model.fit(
    X_train_scaled, y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=2,
)

In [ ]:
# Step 5: Convert the student to TFLite.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd)

print(f"KD student TFLite size: {file_size_kb('model_kd.tflite'):.2f} KB")

In [ ]:
# Step 6: Evaluate the student on the test set.

y_pred_kd = np.argmax(student_model.predict(X_test_scaled, verbose=0), axis=1)
print(f"KD student accuracy: {accuracy_score(y_test, y_pred_kd):.4f}")
print("\nClassification report (KD student):")
print(classification_report(y_test, y_pred_kd, digits=4))
print("Confusion matrix (KD student):")
print(confusion_matrix(y_test, y_pred_kd))

## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [ ]:
# Part (e): Combine knowledge distillation + pruning + full int8 quantization.
#
# Rationale: each technique attacks a different dimension of model size.
#   * KD shrinks the *architecture* (fewer parameters in the student).
#   * Pruning forces many of those parameters to zero (sparse weights).
#   * Int8 quantization shrinks the *per-parameter byte cost* (4 bytes -> 1 byte).
# Stacking them should give a model meaningfully smaller than any single technique,
# while keeping accuracy reasonable thanks to the teacher's soft labels.

prune_epochs_e = 10
end_step_e = int(np.ceil(X_train_scaled.shape[0] / batch_size) * prune_epochs_e)

pruning_schedule_e = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.50,
    final_sparsity=0.70,
    begin_step=0,
    end_step=end_step_e,
)

# Pruned version of the student architecture from Part (d).
student_pruned = Sequential([
    prune_low_magnitude(
        Dense(32, activation='relu', input_shape=(num_features,)),
        pruning_schedule=pruning_schedule_e,
    ),
    prune_low_magnitude(
        Dense(16, activation='relu'),
        pruning_schedule=pruning_schedule_e,
    ),
    prune_low_magnitude(
        Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule_e,
    ),
])

student_pruned.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy'],
)

student_pruned.fit(
    X_train_scaled, y_train_combined,
    epochs=prune_epochs_e,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=2,
)

# Strip pruning wrappers, then int8-quantize through TFLite.
student_pruned_stripped = tfmot.sparsity.keras.strip_pruning(student_pruned)

converter = tf.lite.TFLiteConverter.from_keras_model(student_pruned_stripped)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8
tflite_final = converter.convert()

with open("model_final.tflite", "wb") as f:
    f.write(tflite_final)

print(f"Final (KD + pruning + int8) TFLite size: {file_size_kb('model_final.tflite'):.2f} KB")

# Evaluate the int8 combined model via the TFLite interpreter.
interp = tf.lite.Interpreter(model_path="model_final.tflite")
interp.allocate_tensors()
in_d  = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]

y_pred_final = []
for i in range(len(X_test_scaled)):
    x = X_test_scaled[i:i + 1].astype(np.float32)
    in_scale, in_zp = in_d["quantization"]
    x_q = np.round(x / in_scale + in_zp).astype(np.int8)
    interp.set_tensor(in_d["index"], x_q)
    interp.invoke()
    out = interp.get_tensor(out_d["index"])
    out_scale, out_zp = out_d["quantization"]
    out = (out.astype(np.float32) - out_zp) * out_scale
    y_pred_final.append(int(np.argmax(out)))
y_pred_final = np.array(y_pred_final)

print(f"Final model accuracy: {accuracy_score(y_test, y_pred_final):.4f}")
print("\nClassification report (final combined):")
print(classification_report(y_test, y_pred_final, digits=4))
print("Confusion matrix (final combined):")
print(confusion_matrix(y_test, y_pred_final))

# ------------------------------------------------------------------
# Final comparison across every model from Parts (a)-(e).
# ------------------------------------------------------------------
print("\n=================== Final Size Comparison ===================")
sizes = [
    ("Baseline (float32)",            "model_base.tflite"),
    ("Dynamic range quantization",    "model_dynamic.tflite"),
    ("Full integer (int8)",           "model_int8.tflite"),
    ("Float16 quantization",          "model_float16.tflite"),
    ("Pruned (50%->70%)",             "model_pruned.tflite"),
    ("KD student",                    "model_kd.tflite"),
    ("KD + pruning + int8 (Part e)",  "model_final.tflite"),
]
for label, path in sizes:
    print(f"{label:35s}  {file_size_kb(path):7.2f} KB")

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
